## Machine Learning Feature Engineering for Tree-Based Models

### Objective
Engineer and validate advanced temporal and behavioral features from the unscaled integrated dataset to prepare for predictive modeling using LightGBM and CatBoost.

### Methodology
1. **Data Integrity Checks:** Validate basic sanity (missing values, duplicates) and time-series consistency.
2. **Temporal Aggregation:** Generate 30-day recent behavior metrics (`Orders_last_30d`, `Spend_last_30d`).
3. **Advanced Derived Features:** - Feature Inversion: `Recency_Inv` to align directionality (higher is better).
   - Missingness Flag: `Coupon_Missing` to help tree models differentiate between true zero and lack of data.
   - Interactions: `Value_per_Recency` to capture value decay over time.
   - Normalized Intervals: `Purchase_Interval` smoothed via log transformation.
4. **Export:** Output a rich, multi-dimensional dataset tailored specifically for LightGBM/CatBoost, maintaining native categorical mappings without scaling.

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# load data
sales_df = pd.read_csv(
    "../data/processed/Online_Sales_with_VIP.csv",
    parse_dates=["Transaction_Date"]
)

final_df = pd.read_csv(
    "../data/processed/final_integrated_data_unscaled.csv"
)


In [2]:
# data ranges reasonable?
# basic sanity
# check duplicate CustomerID
assert final_df["CustomerID"].is_unique, "Duplicate CustomerID detected"

# check missing values
df = final_df
missing = df.isnull().sum()
missing = missing[missing > 0]
if missing.empty:
    print("No missing values")
else:
    percent = (missing / len(df)) * 100
    result = pd.DataFrame({
        "MissingCount": missing,
        "MissingPercent": percent
    }).sort_values("MissingPercent", ascending=False)
    display(result)

# check data range
invalid_recency = df[df["Recency"] < 0]
invalid_freq = df[df["Frequency"] < 1]
invalid_monetary = df[df["Monetary"] < 0]
invalid_coupon = df[(df["Coupon_Ratio"] < 0) | (df["Coupon_Ratio"] > 1)]
invalid_tenure = df[df["Tenure_Months"] < 0]
display(df[["Recency", "Frequency", "Monetary", "Coupon_Ratio", "Tenure_Months"]].describe())

No missing values


,Recency,Frequency,Monetary,Coupon_Ratio,Tenure_Months
count,1468.000000,1468.000000,1468.000000,1468.000000,1468.000000
mean,145.292234,18.141008,3560.919407,0.339179,25.912125
std,101.936959,24.976414,5613.006064,0.166418,13.959667
min,1.000000,1.000000,7.000000,0.000000,2.000000
25%,56.000000,5.000000,765.515000,0.264567,14.000000
50%,132.000000,11.000000,1952.680000,0.333333,26.000000
75%,221.000000,23.000000,4362.010000,0.400000,38.000000
max,365.000000,328.000000,83112.260000,1.000000,50.000000


### Sanity check result

No missing values were detected. All numerical features fall within valid ranges, indicating consistent data quality suitable for downstream modeling.

In [3]:
# partial day check
daily_volume = sales_df.groupby(sales_df['Transaction_Date'].dt.date).size()
print(f"daily volume (7 days): {daily_volume.tail(7)}")

# future leakage check
date_columns = sales_df.select_dtypes(include=['datetime64', 'datetimetz']).columns
snapshot_date = sales_df["Transaction_Date"].max() + pd.Timedelta(days=1)
leakage_found = False
for col in date_columns:
    future_leaks = sales_df[sales_df[col] >= snapshot_date]
    if not future_leaks.empty:
        print(f"[WARNING] Column '{col}' contains {len(future_leaks)} rows from the future!")
        leakage_found = True

if not leakage_found:
    print("No future datetime detected beyond the snapshot_date.")

# Time-series Integrity Check
# Sort data chronologically by transaction date
sales_df = sales_df.sort_values('Transaction_Date')
# Calculate the time gap (in days) between consecutive transactions
time_gaps = sales_df['Transaction_Date'].diff().dt.total_seconds() / (24 * 3600)
print("--- Time Gaps Between Transactions (in days) ---")
print(time_gaps.describe())

daily volume (7 days): Transaction_Date
2019-12-25     79
2019-12-26     72
2019-12-27    103
2019-12-28     80
2019-12-29     89
2019-12-30     61
2019-12-31     67
dtype: int64
No future datetime detected beyond the snapshot_date.
--- Time Gaps Between Transactions (in days) ---
count    52923.000000
mean         0.006878
std          0.082648
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          1.000000
Name: Transaction_Date, dtype: float64


### Time Consistency Check

Daily transaction volumes in the most recent period show no abnormal drops or spikes, indicating consistent data recording without partial-day issues.

No future timestamps were detected beyond the snapshot date, confirming the absence of temporal leakage.

Time gap statistics reveal that many transactions occur within the same day or at identical timestamps, resulting in zero or near-zero intervals. This indicates high-frequency transaction behavior rather than strictly daily-level data aggregation.

In [4]:
SNAPSHOT_DATE = sales_df["Transaction_Date"].max() + pd.Timedelta(days=1)
cutoff_30d = SNAPSHOT_DATE - pd.Timedelta(days=30)

# recent behavior features
recent_sales = sales_df[sales_df["Transaction_Date"] >= cutoff_30d]

recent_features = recent_sales.groupby("CustomerID").agg(
    Orders_last_30d=("Transaction_ID", "nunique"),
    Spend_last_30d=("Transaction_Value", "sum")
).reset_index()

enriched_df = pd.merge(final_df, recent_features, on="CustomerID", how="left")

# Fill missing (no activity in last 30 days)
enriched_df["Orders_last_30d"] = enriched_df["Orders_last_30d"].fillna(0)
enriched_df["Spend_last_30d"] = enriched_df["Spend_last_30d"].fillna(0)

print(enriched_df[["Orders_last_30d", "Spend_last_30d"]].describe())


       Orders_last_30d  Spend_last_30d
count      1468.000000     1468.000000
mean          1.837875      372.973399
std           7.035765     1610.037317
min           0.000000        0.000000
25%           0.000000        0.000000
50%           0.000000        0.000000
75%           0.000000        0.000000
max         139.000000    38162.490000


### Recent Behavior Feature Check

The distribution of recent activity features shows a strong zero-inflated pattern, with more than 75% of customers having no transactions within the last 30 days. This indicates a large proportion of inactive users in the recent period.

Both Orders_last_30d and Spend_last_30d exhibit significant right-skewness, where a small subset of customers contributes disproportionately high activity.

This pattern suggests that recent engagement is highly concentrated among a limited group of active customers. The 30-day window captures short-term behavior but may not fully represent activity for less frequent customers.

In [5]:
# DERIVED FEATURES

# Binary indicator for recent activity
enriched_df["Active_30d"] = (enriched_df["Orders_last_30d"] > 0).astype(int)

# Avoid division by zero
enriched_df["Avg_Order_Value"] = np.where(
    enriched_df["Frequency"] > 0,
    enriched_df["Monetary"] / enriched_df["Frequency"],
    0
)

# Purchase interval (days)
DAYS_PER_MONTH = 30.44

enriched_df["Purchase_Interval"] = np.where(
    enriched_df["Frequency"] > 1,
    (enriched_df["Tenure_Months"] * DAYS_PER_MONTH) / enriched_df["Frequency"],
    enriched_df["Tenure_Months"] * DAYS_PER_MONTH
)

print(enriched_df[["Avg_Order_Value", "Purchase_Interval"]].describe())

# clean inf / nan
enriched_df.replace([np.inf, -np.inf], 0, inplace=True)

assert not enriched_df.isna().any().any(), "Missing values remain"

       Avg_Order_Value  Purchase_Interval
count      1468.000000        1468.000000
mean        188.690165         155.597530
std         164.712551         253.968225
min           7.000000           0.908657
25%         130.447540          25.269103
50%         172.674286          63.416667
75%         215.284961         152.200000
max        4481.290000        1522.000000


### Data Cleaning Check

No missing or infinite values remain after preprocessing.

Outlier clipping is not applied, as highly skewed features (e.g., Monetary and Frequency) will be handled through log transformation in the subsequent step, preserving the integrity of extreme but meaningful customer behavior.

In [6]:
features_model = [
    "Recency", "Frequency", "Monetary",
    "Coupon_Ratio", "Tenure_Months",
    "Avg_Order_Value", "Purchase_Interval",
    "Orders_last_30d", "Spend_last_30d",
    "Active_30d"
]

ml_df = enriched_df[["CustomerID"] + features_model]

ml_df.to_csv("../data/processed/ml_features_lightgbm.csv", index=False)